<a href="https://colab.research.google.com/github/Varanapat/2-1_WebPro/blob/main/feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)

In [2]:
# Small epsilon to avoid division by zero
EPS = 1e-6

In [3]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the full suite of OSI feature indicators and attach them to df.

    New columns added
    -----------------
    production_consumption_ratio  : oil_production / oil_consumption
        Range [0, ∞). >1 → net producer.

    import_dependency_ratio       : oil_imports / oil_consumption
        Range [0, 1]. 0 → fully self-sufficient, 1 → fully import-dependent.

    net_import_ratio              : (oil_imports - oil_exports) / oil_consumption
        Range (−∞, 1]. Negative → net exporter (good).

    reserve_consumption_ratio     : oil_reserves_twh / oil_consumption
        Normalised reserve runway in years-equivalent.
        (reserves in bn barrels converted to TWh: 1 bn bbl ≈ 6_117 TWh)

    oil_self_sufficiency_ratio    : min(oil_production, oil_consumption) / oil_consumption
        Range [0, 1]. 1 → fully self-sufficient.

    net_export_score              : oil_exports / (oil_production + EPS)
        Share of production that is exported; positive → net exporter.

    energy_intensity              : oil_consumption / (primary_energy_consumption + EPS)
        Oil's share of total energy. High → oil-dependent economy.

    oil_per_capita                : oil_consumption / population  [TWh per person]

    Returns
    -------
    pd.DataFrame with additional feature columns.
    """
    df = df.copy()

    cons = df["oil_consumption"].clip(lower=EPS)
    prod = df["oil_production"].clip(lower=0)
    imp  = df["oil_imports"].clip(lower=0)
    exp  = df.get("oil_exports", pd.Series(0, index=df.index)).clip(lower=0)


    # ── Core ratios ───────────────────────────────────────────────────────────

    # 1. Production-to-Consumption Ratio
    df["production_consumption_ratio"] = (prod / cons).clip(upper=5.0)

    # 2. Import Dependency Ratio  (capped at 1 — can't import > 100 % of use)
    df["import_dependency_ratio"] = (imp / cons).clip(0, 1)

    # 3. Net Import Ratio
    df["net_import_ratio"] = ((imp - exp) / cons).clip(-2, 1)

    # 4. Reserve-to-Consumption Ratio
    #    Convert reserves from billion barrels → TWh  (1 bn bbl ≈ 6,117 TWh)
    BBL_TO_TWH = 6_117.0
    if "oil_reserves" in df.columns:
        res_twh = df["oil_reserves"].clip(lower=0) * BBL_TO_TWH
    else:
        res_twh = pd.Series(0.0, index=df.index)
    # Express as years-of-consumption; cap at 200 years for normalisation
    df["reserve_consumption_ratio"] = (res_twh / cons).clip(0, 200)

    # 5. Oil Self-Sufficiency Ratio
    df["oil_self_sufficiency_ratio"] = (
        prod.clip(upper=cons) / cons
    ).clip(0, 1)

    # 6. Net Export Score
    df["net_export_score"] = ((prod - cons) / (prod + EPS)).clip(-1, 1)

    # 7. Energy Intensity (oil share of primary energy)
    if "primary_energy_consumption" in df.columns:
        pec = df["primary_energy_consumption"].clip(lower=EPS)
        df["energy_intensity"] = (cons / pec).clip(0, 1)
    else:
        df["energy_intensity"] = np.nan

    # 8. Oil per capita
    if "population" in df.columns:
        pop = df["population"].clip(lower=EPS)
        df["oil_per_capita"] = cons / pop * 1e6   # TWh per million people → MWh/person
    else:
        df["oil_per_capita"] = np.nan

    logger.info("Feature engineering complete.")
    logger.info(f"  New features: {[c for c in df.columns if c not in ['country','year']]}")
    return df


In [4]:
FEATURE_DESCRIPTIONS = {
    "production_consumption_ratio":
        "Ratio of domestic oil production to consumption. "
        ">1 means the country produces more than it consumes (net exporter).",
    "import_dependency_ratio":
        "Share of oil consumption covered by imports. "
        "0 = fully self-sufficient, 1 = fully dependent on imports.",
    "net_import_ratio":
        "Net imports (imports − exports) divided by consumption. "
        "Negative values signal a net exporter.",
    "reserve_consumption_ratio":
        "Proven reserves expressed as years of consumption at current rates. "
        "Higher = more long-term security.",
    "oil_self_sufficiency_ratio":
        "Fraction of consumption met by domestic production (capped at 1). "
        "Used directly in the weighted OSI.",
    "net_export_score":
        "Share of production that is net-exported. "
        "+1 = pure exporter, −1 = full import reliance.",
    "energy_intensity":
        "Oil's share of total primary energy consumption. "
        "High values indicate oil-dependent economies, increasing vulnerability.",
    "oil_per_capita":
        "Oil consumption per person (MWh). "
        "Proxy for economic development and efficiency.",
}